In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
from numpy.ma.core import less_equal
from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
# Create Model
m = gp.Model("Model_1")

Set parameter Username
Set parameter LicenseID to value 2779560
Academic license - for non-commercial use only - expires 2027-02-16


In [3]:
# Time Based GMCNF parameters

# Gravitational acceleration [m/sˆ2]
g_0 = 9.80665

# Number of nodes i,j
nodes = 6

Connections = {0: [0,1] ,
               1: [0,1,2,4],
               2: [1,2,3,4],
               3: [2,3],
               4: [1,2,4,5],
               5: [4,5]}

# Time Steps 12 (days) #testing with +1 day
T = 20

# Advanced Time window
T_adv = list(range(T))
print(T_adv)
#in case multiple time windows are needed this can be added




# Velocity change [km/s]
#this model optimizes for IMLEO,
#  so it does not consider any velocity chage necessary for LEO-PAC (splashdown)
#However, since the model includes splash down, and we want only a single launch per day,
#We can add a Large (big M ) cost for PAC to LEO to ensure that the model does not use this arc
# unless it is really necessary

#Pacific Ocean, Low Earth Orbit, Lunar Lunar Orbit, Lunar Surface, Low Orbit B, Planet B surface
# PAC, LEO, LLO, LS, LOB, PBS are 0, 1, 2, 3, 4 ,5

#Rememeber to add all of the details for necessary holding arcs!
delta_V = {0: {0: 0, 1: 1000}, # PAC to LEO is Big M high
            1: {0: 0, 1: 0, 2: 4.04, 4: 9},
              2: {1: 4.04, 2: 0, 3: 1.87, 4:2},
                3: {2: 1.87, 3: 0},
                    4:{1: 9, 2: 2, 4: 0, 5: 1},
                        5:{4: 1, 5: 0}}

# Time of travel [days]
TOF = {0: {0: 1, 1: 1},
        1: {0: 1, 1: 1, 2: 3, 4: 4},
          2: {1: 3, 2: 1, 3: 1, 4: 2},
            3: {2: 1, 3: 1},
                4: {1: 4, 2: 2, 4: 1, 5: 1},
                    5:{4: 1, 5: 1}}


"""
# I, J = nodes, nodes
# For a general model, the arc routes must be manually defined, since you can only get to certain locations from certain arcs
routes = {i: [j for j in range(nodes) if abs(i - j) <= 1] for i in range(nodes)}

print(routes)
"""

routes = Connections


#WIll be a backup for later in case the routes does not work
def AllpossibleOutflowArcs(Connections, T_adv, connect = routes, TOF = TOF):
    possiblearc = []
    for t in T_adv:
        timepoint = []
        for i in Connections:
            for j in Connections[i]:
                if t+ TOF[i][j] in T_adv:
                    timepoint.append([i,j])
        possiblearc.append(timepoint)
    
    return possiblearc
test = AllpossibleOutflowArcs(Connections,T_adv,routes,TOF)
                
        
        


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [4]:
"""
#Simplified test data


# Number of vehicle types
V = 1



Y = GRB.INTEGER
# Spacecrafts of same type


# Structure mass [kg]
s = np.array([40000])

# Specific impulses [s]
I_sp = np.array([421])

# Payload Capacity [kg]
C = np.array([5000])

# Propellant Capacity [kg]
M = np.array([1200770])
"""


'\n#Simplified test data\n\n\n# Number of vehicle types\nV = 1\n\n\n\nY = GRB.INTEGER\n# Spacecrafts of same type\n\n\n# Structure mass [kg]\ns = np.array([40000])\n\n# Specific impulses [s]\nI_sp = np.array([421])\n\n# Payload Capacity [kg]\nC = np.array([5000])\n\n# Propellant Capacity [kg]\nM = np.array([1200770])\n'

In [5]:
#"""

# Number of vehicle types
V = 2



Y = GRB.INTEGER
# Spacecrafts of same type


#Vehicles: [Saturn V stage 2, Saturn V stage 3, Command Module, Service Module, LM Descent Stage, LM ascent stage]


# Structure mass [kg]
s = np.array([2500, 30000])

# Specific impulses [s]
I_sp = np.array([200, 500])

# Payload Capacity [kg]
C = np.array([1000,75])

# Propellant Capacity [kg] 
M = np.array([65000, 1700000])


#"""

In [6]:
"""
# VEHICLE DATA





# Number of vehicle types
V = 6



Y = GRB.INTEGER
# Spacecrafts of same type


#Vehicles: [Saturn V stage 2, Saturn V stage 3, Command Module, Service Module, LM Descent Stage, LM ascent stage]


# Structure mass [kg]
s = np.array([38415, 12014, 4841, 6053, 2770, 1719])

# Specific impulses [s]
I_sp = np.array([421, 421, 0, 314, 311, 311])

# Payload Capacity [kg]
C = np.array([0, 0, 524, 60, 500, 250])

# Propellant Capacity [kg] 
M = np.array([452045, 107725, 0, 18413, 8804, 2358])

"""







'\n# VEHICLE DATA\n\n\n\n\n\n# Number of vehicle types\nV = 6\n\n\n\nY = GRB.INTEGER\n# Spacecrafts of same type\n\n\n#Vehicles: [Saturn V stage 2, Saturn V stage 3, Command Module, Service Module, LM Descent Stage, LM ascent stage]\n\n\n# Structure mass [kg]\ns = np.array([38415, 12014, 4841, 6053, 2770, 1719])\n\n# Specific impulses [s]\nI_sp = np.array([421, 421, 0, 314, 311, 311])\n\n# Payload Capacity [kg]\nC = np.array([0, 0, 524, 60, 500, 250])\n\n# Propellant Capacity [kg] \nM = np.array([452045, 107725, 0, 18413, 8804, 2358])\n\n'

In [7]:
# COMMODITY Data and Demand/Supply

# Propellant mass fraction
#defined from rocket equation 1-e**(-deltav/Ispg0)

#actually the official function is e**(-deltav/Ispg0), but using the 1-e form allows use 
# to multiply the contents with the unchanging masses to include their input into the transformation linearly
#This varies with the delta v necessary for each arc, and the Isp of the vehicle used for that arc
#Will be used later


def phi(i,j,v, dV = delta_V, I_sp = I_sp, g_0 = g_0):
    if I_sp[v] == 0:
        return 1
    else:
        return 1 - np.exp(-(1000*dV[i][j] / (I_sp[v] * g_0))) #1000 used for conversion



# Commodity variable types
# Crew, consumables kg, equipment kg, samples kg, propellant kg
X = [GRB.INTEGER, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.CONTINUOUS]
PropIndex = 4

# Crew mass [kg/crew]
crew_mass = 100


CommodityMassConversion = [crew_mass,1,1,1,1]

# Crew, consumables kg, equipment kg, samples kg, propellant kg
# PAYLOAD ASSUMPTIONS

# Consumption rates [kg/crew/day]
food_consumption = 1.0
water_consumption = 5.0
oxygen_consumption = 1.1
consumption = food_consumption + water_consumption + oxygen_consumption
#if the model works, this can be made more granular by separating the consumptions


# Upper limit of each variable per spacecraft is the capacity of each spacecraft *number of spacecraft in that node.
# Crew, consumables kg, equipment kg, samples kg, propellant kg
XUpper = [C/crew_mass, C, C, C, M]

#Single Spacecraft consumption table (from outflow to inflow consumption of all commodities)

#Matrix multiplication with a vector of outflows


# There is no propellant output
def Solo_SC_Consumption(i, j,v, consumption = consumption, TOF = TOF,structure_mass = s):

    # Crew, consumables kg, equipment kg, samples kg, propellant kg, number of spacecraft
    value = np.array([[1, 0, 0, 0, 0, 0], #Crew
                        [-consumption*TOF[i][j] , 1, 0, 0, 0, 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0], #Equipment
                        [0, 0, 0, 1, 0, 0], #Samples
                        [crew_mass * -1*phi(i,j,v,delta_V,I_sp,g_0),
                       -1*phi(i,j,v,delta_V,I_sp,g_0),
                         -1*phi(i,j,v,delta_V,I_sp,g_0),
                           -1*phi(i,j,v,delta_V,I_sp,g_0),
                             1-1*phi(i,j,v,delta_V,I_sp,g_0),
                                -1*structure_mass[v]*phi(i,j,v,delta_V,I_sp,g_0)], #Propellant
                        [0, 0, 0, 0, 0, 1]])# last row (and column) is for number of spacecraft
    
    return value

def Solo_SC_Consumption_NodV(i, j,v, consumption = consumption, TOF = TOF,structure_mass = s):

    # Crew, consumables kg, equipment kg, samples kg, propellant kg, number of spacecraft
    value = np.array([[1, 0, 0, 0, 0, 0], #Crew
                        [-consumption*TOF[i][j] , 1, 0, 0, 0, 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0], #Equipment
                        [0, 0, 0, 1, 0, 0], #Samples
                        [0, 0, 0, 0, 1, 0], #Propellant
                        [0, 0, 0, 0, 0, 1]])# last row (and column) is for number of spacecraft
    
    return value








# THE COMMODITIES ARE PROVIDED AT LEO. START ALL THINGS AT LEO
# CHECK DAYS FOR MISSION !!!
#Commodity Array: D[Node][Day][Commodity]



D = [[np.array([0 for x in range(len(X))])
      for _ in T_adv]
    for _ in routes]

print(len(D))
print(len(D[0]))
print(len(D[0][0]))

#initial test values
# Earth (PAC) consumables, equipment, propellant supply infinite at leo time 0, AND Moon surface sample supply (infinite at all times)
#Crew are capped in supply so we don't leave anyone on the moon
D[1][0][0] = 3 #Crew
D[1][0][1] = 99999 #Consumables
D[1][0][2] = 99999 #Equipment
D[1][0][4] = 99999999999 #Propellant

for x in T_adv:
    D[3][x][3] = 999999 #Moon samples


# APOLLO
# Remember we are using a list starting at 0
# Crew demand/supply
D[3][4][0] = -2 # Lunar surface day 5 crew demand (negative supply)
D[2][3][0] = -1 # Lunar orbit day 4 crew demand
D[3][5][0] = 2 # Lunar surface day 6 crew supply (return)
D[2][6][0] = 1 # Lunar orbit day 7 crew supply (return)
D[0][19][0] = -3 # Earth day 11 crew demand (return)

D[3][4][2] = -420 # Lunar surface day 5 (scientific) equipment demand

D[0][19][3] = -110 # Earth day 11 lunar sample demand

#Extra
D[5][10][2] = -200 #Equipment Demand on Planet B
D[5][11][0] = -1 # Astronaut Demand on Planet B
D[5][13][0] = 1 # Astronaut Return Demand on Planet B

# S/C COMMODITY DEMAND
# format d[node][vehicle][day]
# Infinite supply of SC at LEO day 1 (time 0), none elsewhere
d = [[[5 if (i == 1 and t == 0) else 0 for t in range(T)] # Infinite supply of spacecrafts at i = 1, t = 0 LEO
     for _ in range(V)]
    for i in routes]


6
20
5


In [8]:
# CREATE COMMODITY FLOW VECTORS AND S/C COMMODITY FLOW
# Variable naming process: commodity_{direction}flow_{v},{i},{j},{t},{x}'
#v: vehicle type, i: node of origin, j: node of destination, t: time step, x: commodity type
# Spacecraft Variable naming process: sc_commodity_{direction}flow_{v},{i},{j},{t}'




#Only create variables for arcs that end within the destination window.

#TOF tells you the travel times
def check_destination_window(startnode, endnode, tstart, All_nodes= T_adv, TOF = TOF):
    print(startnode)
    print(endnode)
    arrival =  TOF[startnode][endnode]+tstart

    if arrival in All_nodes:
        return True
    else:
        return False


#lower bound for all commodities is 0, no negatives.
def create_commodity_flow(model, V, X, Time = T_adv, direction = "out", connect = routes, typeC = "Classic"):

    
    
    x_flow = [[{j: [np.array([[model.addVar(vtype=X[x], name=f'{typeC}_commodity_{direction}flow_{v},{i},{j},{t},{x}',lb = 0 )]
                              for x in range(len(X))])
                    for t in range(T-1) if check_destination_window(i, j, t, Time, TOF)]
                for j in connect[i]}
               for i in connect ]
              for v in range(V)]


    return x_flow


def create_sc_commodity_flow(model, V,Y, Time = T_adv, direction = "out", connect = routes):
    y_flow = [[{j: [np.array([model.addVar(vtype=Y, name=f'sc_commodity_{direction}flow_{v},{i},{j},{t}',lb=0)])
            for t in range(T-1) if check_destination_window(i, j, t, Time, TOF)]
          for j in connect[i]}
         for i in connect]
        for v in range(V)]

    return y_flow

# Outflow+ leaving from node i to j, inflow- arriving at node j from i

x_outflow, x_inflow = create_commodity_flow(m, V, X, T_adv, direction="out", connect=routes), create_commodity_flow(m, V, X, T_adv, direction="in", connect=routes)
y_outflow, y_inflow = create_sc_commodity_flow(m, V, Y, T_adv, direction="out", connect=routes), create_sc_commodity_flow(m, V, Y, T_adv, direction="in", connect=routes)






m.update()

#print(y_outflow[0][0][0][-1])
#print(y_outflow[0][0][0][-2])
#print(y_outflow[0][0][0][-3])




0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
2
4
2
4
2


In [9]:
# ADD THE CONSTRAINTS (2 & 3)
# CONSTRAINTS 2 & 3 MASS BALANCE
# Node commodity demand D vectors (positive for supply)
# sum(x[i][t]+) - sum(x[i][t]-) <= D[i][t]

import sys

#this is done for all nodes except the final day

for i in routes:
    for t in T_adv: #range(T - 1)) also works for simple tests

        
        
        
        x_outflow_sum = sum(x_outflow[v][i][j][t] 
                            if t +TOF[i][j] in T_adv
                            else np.array([[0] for _ in range(len(X))])  #packaged SC are dealt with differently
                            for v in range(V) for j in routes[i])
            #)
        # On the last day there is no outflow, the if else statement ensures
        #that only the outflows for which the spacecraft has had time to arrive are counted
    

        x_inflow_sum = sum(x_inflow[v][j][i][t - TOF[j][i]] 
                           if t - TOF[j][i] in T_adv
                           else np.array([[0] for _ in range(len(X))])
                           for v in range(V) for j in routes[i])
        # Only count the inflows for which the spacecraft has had time to arrive 
        # (or where it has had time to depart (no negative times))


        #if t == 0:
        #    print(x_outflow_sum)
        #    print(len(x_outflow_sum))
        #    print(x_inflow_sum)
        #    print(len(x_inflow_sum[0]))
        #    sys.exit()

        for x in range(len(X)):
            try:
                m.addConstr(x_outflow_sum[x][0] - x_inflow_sum[x][0] <= D[i][t][x],
                            name=f"mass_balance_x_node{i}_time{t}_comm{x}")
                #print(x_outflow_sum[x][0])
            
            except Exception as e:
                print(f"Error on node={i}, time={t}, commodity={x}: {e}")
                raise
            
            #except:
            #    print("Error on constraint for node {}, time {}, commodity {}".format(i, t, x))

        # S/C commodity supply and demand
        for v in range(V):
            y_outflow_sum = sum(y_outflow[v][i][j][t]  
                                if t +TOF[i][j] in T_adv \
                                else np.array([0]) 
                                for j in routes[i])

            y_inflow_sum = sum(y_inflow[v][j][i][t - TOF[j][i]] 
                               if t - TOF[j][i] in T_adv \
                               else np.array([0])
                               for j in routes[i])

                

            #temporary pick and choose, change this later, only sinks at the end of the model
            if (t < T-1) and (t != 0):

                m.addConstr(y_outflow_sum[0] - y_inflow_sum[0] == d[i][v][t],
                name=f"SC_lossless_mass_balance_x_node{i}_time{t}_vehicle{v}") #lose no SC until the end
            else:
                m.addConstr(y_outflow_sum[0] - y_inflow_sum[0] <= d[i][v][t],
                name=f"SC_mass_balance_x_node{i}_time{t}_vehicle{v}")


        
        


m.update()

In [10]:
## Commodity transformation
#import sys


for i in routes:
    for j in routes[i]:
        for t in T_adv:

             if check_destination_window(i,j,t,T_adv,TOF):
                  
                for v in range(V):
                
                
                    #print(i,j,t,v)
                
                
                    Vout = np.concatenate((x_outflow[v][i][j][t],
                        np.array([y_outflow[v][i][j][t]])), axis=0)


                    Vin = np.concatenate((x_inflow[v][i][j][t],
                         np.array([y_inflow[v][i][j][t]])), axis=0)
                    




                    #print(Vin)
                    #sys.exit()

                    
                    #create the correct consumption matrix, based on deltav and travel time
                    if delta_V[i][j] == 0: #If there is no Delta v: there is no propellant consumption
                        Consumed =Solo_SC_Consumption_NodV(i,j,v,consumption,TOF,s)
                    
                    else:
                        Consumed =Solo_SC_Consumption(i,j,v,consumption,TOF,s)
                    
                    

                    


                    transformed = np.dot(Consumed,Vout)
                
                    
                    for i1,(enterarc,leavearc) in enumerate(zip(transformed, Vin)):

                        m.addConstr(enterarc[0] == leavearc[0],name=f'Arc_transformationConstraint_Start{i}_End{j}_Starttime{t}_Vehicle{v}_Commodity{i1}')

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1


In [11]:
# CONSTRAINTS 5 CONCURRENCY LIMITS

# Concurrency constraint matrix
# H[x+] <= e * y+ --> Payload mass and fuel in s/c does not exceed maximum capacities
# x = Crew, consumables, equipment, samples, propellant

# def create_concurrency_constraint(V, connect =routes, crew_mass): # Different vehicles version
#     H = [[{j: np.array([[crew_mass, 1, 1, 1, 0],
#                         [0, 0, 0, 0, 1]])
#            for j in connect[i]}
#           for i in connect]
#          for v in range(V)]
#
#     return H


def create_concurrency_constraint(connect = routes, crew_mass = crew_mass): # Same for all vehicles, max payload mass
    H = [{j: np.array([[crew_mass, 1, 1, 1, 0], #payload
                        [0, 0, 0, 0, 1]]) #Propellant
           for j in connect[i]}
          for i in connect]
    return H


def create_sc_design_parameters(V, C, M):
    e = [np.array([[C[v]], [M[v]]]) for v in range(V)]
    return e


H = create_concurrency_constraint(routes, crew_mass)
e = create_sc_design_parameters(V, C, M)
print(H)
print(len(H))

[{0: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 1: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]])}, {0: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 1: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 2: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 4: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]])}, {1: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 2: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 3: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 4: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]])}, {2: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 3: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]])}, {1: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 2: array([[100,   1,   1,   1,   0],
    

In [12]:
# ADD THE CONSTRAINTS (5)

for v in range(V):
    for i in routes:
        for j in routes[i]:
            for t in range(T-1):
                if check_destination_window(i,j,t,T_adv,TOF):

                    for commodity, constraint in zip(np.dot(H[i][j], x_outflow[v][i][j][t]),
                                                     e[v]*y_outflow[v][i][j][t][0]):

                        m.addConstr(commodity[0] <= constraint[0])

m.update()


0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
2
4
2
4
2


In [13]:
# CONSTRAINTS 6 TIME-WINDOW
# ADD THE CONSTRAINTS (6)

#Minimum value of 0 for all arcs

for v in range(V):
    for i in routes:
        for j in routes[i]:
            for t in range(T-1):
                if check_destination_window(i,j,t,T_adv,TOF):

                    for commodity_out in x_outflow[v][i][j][t]:
                        m.addConstr(commodity_out[0] >= 0)

                    for commodity_in in x_inflow[v][i][j][t]:
                        m.addConstr(commodity_in[0] >= 0)

                    m.addConstr(y_outflow[v][i][j][t][0] >= 0)
                    m.addConstr(y_inflow[v][i][j][t][0] >= 0)

m.update()

# s[v] >= 0


0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
0
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
1
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
2
4
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
2
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
1
4
2
4
2
4
2


In [14]:
# CONSTRAINTS 7 SPACE-CRAFT MASS

In [15]:
# COST FUNCTION - INITIAL MASS AT LEO
# sum(cost * x + cost_y * s * y)

# x = Crew, consumables, equipment, samples, propellant, crew(return)

#Currently the model assumes we are starting at the LEO node t = 0, i=1

def create_commodity_cost(V, connect = routes, crew_mass= crew_mass):
    cost_coeff = [[{j:
                        [np.array([[crew_mass], [1], [1], [1], [1]]) if (t == 0 and i == 1)
                         else np.array([[0] for _ in range(len(X))])
                         for t in range(T-1)]
           for j in connect[i]}
          for i in connect]
         for _ in range(V)]

    sc_cost_coeff = [[{j:
                        [1 if (t == 0 and i == 1)
                         else 0
                         for t in range(T-1)]
           for j in connect[i]}
          for i in connect]
         for _ in range(V)]

    return cost_coeff, sc_cost_coeff

cost_coeff, sc_cost_coeff = create_commodity_cost(V, routes, crew_mass)

In [16]:
# DEFINE THE COST FUNCTION (1)

"""
#General version
cost = sum(
    np.dot(cost_coeff[v][i][j][t].T, x_outflow[v][i][j][t]) + sc_cost_coeff[v][i][j][t] * s[v] * y_outflow[v][i][j][t][0]
    for v in range(V)
    for i in routes
    for j in routes[i]
    for t in range(3)
)
"""
#specific to apollo version
#only looking at node 1 at time t = 0
cost = sum(
    np.dot(cost_coeff[v][1][j][0].T, x_outflow[v][1][j][0]) + sc_cost_coeff[v][1][j][0] * s[v] * y_outflow[v][1][j][0][0]
    for v in range(V)
    for j in routes[1]
)

cost = cost[0][0]
print(cost)

m.setObjective(cost, GRB.MINIMIZE)
m.update()

100.0 Classic_commodity_outflow_0,1,0,0,0 + Classic_commodity_outflow_0,1,0,0,1 + Classic_commodity_outflow_0,1,0,0,2 + Classic_commodity_outflow_0,1,0,0,3 + Classic_commodity_outflow_0,1,0,0,4 + 2500.0 sc_commodity_outflow_0,1,0,0 + 100.0 Classic_commodity_outflow_0,1,1,0,0 + Classic_commodity_outflow_0,1,1,0,1 + Classic_commodity_outflow_0,1,1,0,2 + Classic_commodity_outflow_0,1,1,0,3 + Classic_commodity_outflow_0,1,1,0,4 + 2500.0 sc_commodity_outflow_0,1,1,0 + 100.0 Classic_commodity_outflow_0,1,2,0,0 + Classic_commodity_outflow_0,1,2,0,1 + Classic_commodity_outflow_0,1,2,0,2 + Classic_commodity_outflow_0,1,2,0,3 + Classic_commodity_outflow_0,1,2,0,4 + 2500.0 sc_commodity_outflow_0,1,2,0 + 100.0 Classic_commodity_outflow_0,1,4,0,0 + Classic_commodity_outflow_0,1,4,0,1 + Classic_commodity_outflow_0,1,4,0,2 + Classic_commodity_outflow_0,1,4,0,3 + Classic_commodity_outflow_0,1,4,0,4 + 2500.0 sc_commodity_outflow_0,1,4,0 + 100.0 Classic_commodity_outflow_1,1,0,0,0 + Classic_commodity_ou

In [17]:
m.optimize()

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i9-10885H CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 14040 rows, 7920 columns and 30972 nonzeros
Model fingerprint: 0xa42bd0a3
Variable types: 5280 continuous, 2640 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e-02, 2e+06]
  Objective range  [1e+00, 3e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+11]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Presolve removed 12712 rows and 5774 columns
Presolve time: 0.06s
Presolved: 1328 rows, 2146 columns, 7849 nonzeros
Variable types: 1409 continuous, 737 integer (0 binary)

Root relaxation: objective 1.184773e+06, 3109 iterations, 0.30 seconds (0.18 work units)

    Nodes    |    Current Node    |     Objective Bounds      |    

In [18]:

"""
# Good practice: write the full model first
m.update()
m.write("debug_model_new.lp")

m.optimize()

# If Gurobi says infeasible or unbounded, resolve with DualReductions off
if m.Status == GRB.INF_OR_UNBD:
    m.Params.DualReductions = 0
    m.optimize()

if m.Status == GRB.INFEASIBLE:
    print("Model is infeasible. Computing IIS...")
    m.computeIIS()

    # Writes a small model containing the infeasible subsystem
    m.write("infeasible_subset.ilp")

    print("\nConstraints in IIS:")
    for c in m.getConstrs():
        if c.IISConstr:
            print(f"{c.ConstrName}: sense={c.Sense}, RHS={c.RHS}")

    print("\nVariable bounds in IIS:")
    for v in m.getVars():
        if v.IISLB:
            print(f"{v.VarName}: lower bound {v.LB}")
        if v.IISUB:
            print(f"{v.VarName}: upper bound {v.UB}")

#"""

'\n# Good practice: write the full model first\nm.update()\nm.write("debug_model_new.lp")\n\nm.optimize()\n\n# If Gurobi says infeasible or unbounded, resolve with DualReductions off\nif m.Status == GRB.INF_OR_UNBD:\n    m.Params.DualReductions = 0\n    m.optimize()\n\nif m.Status == GRB.INFEASIBLE:\n    print("Model is infeasible. Computing IIS...")\n    m.computeIIS()\n\n    # Writes a small model containing the infeasible subsystem\n    m.write("infeasible_subset.ilp")\n\n    print("\nConstraints in IIS:")\n    for c in m.getConstrs():\n        if c.IISConstr:\n            print(f"{c.ConstrName}: sense={c.Sense}, RHS={c.RHS}")\n\n    print("\nVariable bounds in IIS:")\n    for v in m.getVars():\n        if v.IISLB:\n            print(f"{v.VarName}: lower bound {v.LB}")\n        if v.IISUB:\n            print(f"{v.VarName}: upper bound {v.UB}")\n\n#'

In [19]:
import sys
sys.path.append("/2026_code/")  # needed to add in the results file
import Results

m.update()
m.write("debug_model_Simple.lp")

Apollo_Veh =[
        "Saturn V S-II",
        "Saturn V S-IVB",
        "Command Module",
        "Service Module",
        "LM Descent Stage",
        "LM Ascent Stage"
    ]

Two_ship_test =["Payload","Fuel"]

namevar = Two_ship_test

Cargoflows,Shipflows = Results.extract_flows(
    x_outflow=x_outflow,
    x_inflow=x_inflow,
    y_outflow=y_outflow,
    arcs=routes,
    node_names=["PAC", "LEO", "LLO", "LS","LOB", "PBS"],
    vehicle_names=namevar,
    commodity_names=[
        "crew",
        "consumables",
        "equipment",
        "sample",
        "propellant"
    ],
    tof=TOF,
    T=T,
    T_adv = T_adv,
    CommMass = CommodityMassConversion
)

#flows.head()


Results.plot_time_space_network(
    Shipflows,
    Cargoflows,
    node_order=["PAC", "LEO", "LLO", "LS","LOB","PBS"],
    title="Apollo 17 optimized time-space logistics solution"
)

mass_table = Results.make_mass_flow_table(Cargoflows, use="out_mass")
#mass_table

mass_table.style.format(precision=1)

Results.propellantUsage(Cargoflows)

Results.plot_vehicle_gantt(Cargoflows, title="Apollo 17 spacecraft activity")

In [20]:
mass_table.style.format(precision=1)

commodity,t_depart,t_arrive,from_node,to_node,vehicle,n_ships,consumables,crew,equipment,propellant,sample,total_flow
0,0,3,LEO,LLO,Fuel,2.0,150.0,0.0,0.0,1341927.2,0.0,1342077.2
1,0,3,LEO,LLO,Payload,2.0,205.0,300.0,620.0,41923.2,0.0,43048.2
2,3,4,LLO,LLO,Fuel,2.0,0.0,0.0,0.0,538241.4,0.0,538241.4
3,3,4,LLO,LLO,Payload,1.0,262.7,0.0,200.0,0.0,0.0,462.7
4,3,4,LLO,LS,Payload,1.0,28.4,200.0,420.0,16705.2,0.0,17353.6
5,4,5,LLO,LLO,Payload,1.0,262.7,0.0,50.0,24480.9,0.0,24793.6
7,4,6,LLO,LOB,Fuel,2.0,0.0,0.0,150.0,513760.5,0.0,513910.5
6,4,5,LS,LS,Payload,1.0,14.2,0.0,0.0,4503.5,0.0,4517.7
8,5,6,LLO,LLO,Payload,1.0,262.7,0.0,50.0,24480.9,0.0,24793.6
9,5,6,LS,LLO,Payload,1.0,14.2,200.0,0.0,4503.5,110.0,4827.7


In [21]:
Shipflows.head()

,vehicle,v,from_node,to_node,i,j,t_depart,t_arrive,n_ships
0,Payload,0,LEO,PAC,1,0,18,19,1.0
1,Payload,0,LEO,LLO,1,2,0,3,2.0
2,Payload,0,LLO,LEO,2,1,15,18,1.0
3,Payload,0,LLO,LLO,2,2,3,4,1.0
4,Payload,0,LLO,LLO,2,2,4,5,1.0


In [22]:
# Print the values of all variables
for v in m.getVars():
    print(f"{v.VarName} = {v.X}")

Classic_commodity_outflow_0,0,0,0,0 = 0.0
Classic_commodity_outflow_0,0,0,0,1 = 0.0
Classic_commodity_outflow_0,0,0,0,2 = 0.0
Classic_commodity_outflow_0,0,0,0,3 = 0.0
Classic_commodity_outflow_0,0,0,0,4 = 0.0
Classic_commodity_outflow_0,0,0,1,0 = -0.0
Classic_commodity_outflow_0,0,0,1,1 = 0.0
Classic_commodity_outflow_0,0,0,1,2 = 0.0
Classic_commodity_outflow_0,0,0,1,3 = 0.0
Classic_commodity_outflow_0,0,0,1,4 = 0.0
Classic_commodity_outflow_0,0,0,2,0 = -0.0
Classic_commodity_outflow_0,0,0,2,1 = 0.0
Classic_commodity_outflow_0,0,0,2,2 = 0.0
Classic_commodity_outflow_0,0,0,2,3 = 0.0
Classic_commodity_outflow_0,0,0,2,4 = 0.0
Classic_commodity_outflow_0,0,0,3,0 = -0.0
Classic_commodity_outflow_0,0,0,3,1 = 0.0
Classic_commodity_outflow_0,0,0,3,2 = 0.0
Classic_commodity_outflow_0,0,0,3,3 = 0.0
Classic_commodity_outflow_0,0,0,3,4 = 0.0
Classic_commodity_outflow_0,0,0,4,0 = -0.0
Classic_commodity_outflow_0,0,0,4,1 = 0.0
Classic_commodity_outflow_0,0,0,4,2 = 0.0
Classic_commodity_outflow_0,0,

In [23]:
"""

# x = Crew, consumables, equipment, samples, propellant
#Variable naming process: commodity_{direction}flow_{v},{i},{j},{t},{x}'
# Variable naming process: sc_commodity_{direction}flow_{v},{i},{j},{t}'
import pandas as pd

results = {final_variable.VarName: final_variable.X for final_variable in m.getVars()}

sorted_results = dict(sorted(results.items(), key=lambda item: int(item[0].split(",")[3])))

f = open("MostBasic.txt", "w")



for final in sorted_results:
    if sorted_results[final] != 0:
        print('%s %g' % (final, sorted_results[final]))
        f.write('%s %g' % (final, sorted_results[final]))
        f.write('\n')
f.close()
"""

'\n\n# x = Crew, consumables, equipment, samples, propellant\n#Variable naming process: commodity_{direction}flow_{v},{i},{j},{t},{x}\'\n# Variable naming process: sc_commodity_{direction}flow_{v},{i},{j},{t}\'\nimport pandas as pd\n\nresults = {final_variable.VarName: final_variable.X for final_variable in m.getVars()}\n\nsorted_results = dict(sorted(results.items(), key=lambda item: int(item[0].split(",")[3])))\n\nf = open("MostBasic.txt", "w")\n\n\n\nfor final in sorted_results:\n    if sorted_results[final] != 0:\n        print(\'%s %g\' % (final, sorted_results[final]))\n        f.write(\'%s %g\' % (final, sorted_results[final]))\n        f.write(\'\n\')\nf.close()\n'

In [24]:
# # EQUATION 7 CONSTRAINTS
#
# # Structural Fraction (fuel dependent)
# alpha = 0.045  # LOX/kerosene
#
# # Gravitational Acceleration Earth
# g_0 = 9.8  # m/s2
#
# # Upper Bound Allowed for Propellant Tank Capacity
# M_ub = 500000  # kg
#
# # Spacecraft Impulsive Burn
# t_b = 120  # s
#
#
# # Structure Mass Variable
# def create_s_star_variables(model, v=V):
#     variables = {}
#     for v in range(V):
#         variables[v] = model.addVar(vtype=GRB.CONTINUOUS, name=f'Structure_Mass_{v}')
#     return variables
#
#
# s_star = create_s_star_variables(model=m)
#
# m.update()

In [25]:
# # CONSTRAINTS 7
#
# for v in tqdm(V):
#     m.addConstr(s_star[v] = 2.3931 * )